# Cross-Dataset Evaluation and Transfer Learning

This notebook demonstrates how to evaluate text classification models across multiple datasets
and implement transfer learning techniques for improved generalization.

## Learning Objectives
1. Understand cross-dataset evaluation challenges
2. Implement transfer learning for text classification
3. Analyze domain adaptation techniques
4. Compare model performance across different datasets
5. Apply XAI techniques to understand transfer learning

In [ ]:
# Import required libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Import project modules
from src.dataset_integration import DatasetManager
from src.explainable_ai import extract_feature_importance, visualize_feature_importance

# ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# Set style
plt.style.use('default')
sns.set_palette('husl')

print('✅ All libraries imported successfully!')
print(f'Project root: {project_root}')

## 1. Dataset Loading and Exploration

Let's load multiple datasets and explore their characteristics.

In [ ]:
# Initialize dataset manager
dataset_manager = DatasetManager()

# List available datasets
available_datasets = ['jigsaw_toxic', 'ag_news', 'imdb', 'twitter_hate', 'fake_news']
print('Available datasets:', available_datasets)

# Load dataset information
dataset_info = {}
for dataset_name in available_datasets:
    info = dataset_manager.get_dataset_info(dataset_name)
    dataset_info[dataset_name] = info
    print(f'\n{dataset_name.upper()}:')
    for key, value in info.items():
        print(f'  {key}: {value}')

In [ ]:
# Load datasets with smaller sample sizes for demonstration
datasets = {}
sample_size = 500  # Adjust based on computational resources

for dataset_name in available_datasets:
    print(f'Loading {dataset_name}...')
    try:
        train_df, test_df = dataset_manager.load_dataset(dataset_name, sample_size=sample_size)
        
        # Standardize format
        train_df = dataset_manager.standardize_dataset(train_df, dataset_name)
        test_df = dataset_manager.standardize_dataset(test_df, dataset_name)
        
        datasets[dataset_name] = {
            'train': train_df,
            'test': test_df
        }
        
        print(f'  Train: {train_df.shape}, Test: {test_df.shape}')
        print(f'  Labels: {train_df["label"].unique()}')
        
    except Exception as e:
        print(f'  Error loading {dataset_name}: {e}')

print(f'\n✅ Loaded {len(datasets)} datasets successfully!')

## 2. Dataset Characteristics Analysis

Let's analyze the characteristics of each dataset to understand their differences.

In [ ]:
# Analyze text characteristics across datasets
def analyze_text_characteristics(datasets):
    """Analyze text characteristics across multiple datasets."""
    characteristics = []
    
    for dataset_name, data in datasets.items():
        train_df = data['train']
        
        # Calculate text statistics
        text_lengths = train_df['text'].str.len()
        word_counts = train_df['text'].str.split().str.len()
        
        char_stats = {
            'dataset': dataset_name,
            'samples': len(train_df),
            'avg_char_length': text_lengths.mean(),
            'std_char_length': text_lengths.std(),
            'avg_word_count': word_counts.mean(),
            'std_word_count': word_counts.std(),
            'label_distribution': train_df['label'].value_counts().to_dict()
        }
        
        characteristics.append(char_stats)
    
    return pd.DataFrame(characteristics)

# Analyze characteristics
char_df = analyze_text_characteristics(datasets)
print('Dataset Characteristics:')
print(char_df[['dataset', 'samples', 'avg_char_length', 'avg_word_count']].round(2))

In [ ]:
# Visualize dataset characteristics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Text length distribution
axes[0, 0].bar(char_df['dataset'], char_df['avg_char_length'])
axes[0, 0].set_title('Average Character Length by Dataset')
axes[0, 0].set_ylabel('Characters')
axes[0, 0].tick_params(axis='x', rotation=45)

# Word count distribution
axes[0, 1].bar(char_df['dataset'], char_df['avg_word_count'])
axes[0, 1].set_title('Average Word Count by Dataset')
axes[0, 1].set_ylabel('Words')
axes[0, 1].tick_params(axis='x', rotation=45)

# Sample distribution
axes[1, 0].bar(char_df['dataset'], char_df['samples'])
axes[1, 0].set_title('Sample Count by Dataset')
axes[1, 0].set_ylabel('Samples')
axes[1, 0].tick_params(axis='x', rotation=45)

# Text length vs word count scatter
axes[1, 1].scatter(char_df['avg_char_length'], char_df['avg_word_count'])
for i, dataset in enumerate(char_df['dataset']):
    axes[1, 1].annotate(dataset, 
                        (char_df['avg_char_length'].iloc[i], 
                         char_df['avg_word_count'].iloc[i]))
axes[1, 1].set_xlabel('Average Character Length')
axes[1, 1].set_ylabel('Average Word Count')
axes[1, 1].set_title('Character Length vs Word Count')

plt.tight_layout()
plt.show()

## 3. Cross-Dataset Model Training

Train models on one dataset and evaluate on others to understand transfer performance.

In [ ]:
class CrossDatasetEvaluator:
    """Evaluate models across multiple datasets."""
    
    def __init__(self, datasets):
        self.datasets = datasets
        self.models = {}
        self.vectorizers = {}
        self.results = {}
    
    def train_model(self, source_dataset, model_type='logistic'):
        """Train a model on the source dataset."""
        train_df = self.datasets[source_dataset]['train']
        
        # Initialize vectorizer
        vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        )
        
        # Vectorize text
        X_train = vectorizer.fit_transform(train_df['text'])
        y_train = train_df['label']
        
        # Initialize model
        if model_type == 'logistic':
            model = LogisticRegression(random_state=42, max_iter=1000)
        elif model_type == 'random_forest':
            model = RandomForestClassifier(n_estimators=100, random_state=42)
        elif model_type == 'svm':
            model = SVC(kernel='linear', probability=True, random_state=42)
        else:
            model = MultinomialNB()
        
        # Train model
        model.fit(X_train, y_train)
        
        # Store model and vectorizer
        model_key = f'{source_dataset}_{model_type}'
        self.models[model_key] = model
        self.vectorizers[model_key] = vectorizer
        
        print(f'✅ Trained {model_type} model on {source_dataset}')
        return model_key
    
    def evaluate_cross_dataset(self, model_key, target_dataset):
        """Evaluate a trained model on a target dataset."""
        model = self.models[model_key]
        vectorizer = self.vectorizers[model_key]
        
        # Get target dataset
        test_df = self.datasets[target_dataset]['test']
        
        # Vectorize target text
        X_test = vectorizer.transform(test_df['text'])
        y_test = test_df['label']
        
        # Make predictions
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        
        return {
            'accuracy': accuracy,
            'predictions': y_pred,
            'probabilities': y_proba,
            'true_labels': y_test
        }
    
    def run_cross_evaluation(self, model_types=['logistic']):
        """Run comprehensive cross-dataset evaluation."""
        results = {}
        
        for source_dataset in self.datasets.keys():
            for model_type in model_types:
                # Train model on source dataset
                model_key = self.train_model(source_dataset, model_type)
                
                # Evaluate on all datasets
                for target_dataset in self.datasets.keys():
                    eval_key = f'{source_dataset}_to_{target_dataset}_{model_type}'
                    results[eval_key] = self.evaluate_cross_dataset(model_key, target_dataset)
                    
                    accuracy = results[eval_key]['accuracy']
                    print(f'{source_dataset} → {target_dataset} ({model_type}): {accuracy:.4f}')
        
        self.results = results
        return results

# Initialize evaluator
evaluator = CrossDatasetEvaluator(datasets)
print('Cross-dataset evaluator initialized!')

In [ ]:
# Run cross-dataset evaluation
print('Running cross-dataset evaluation...')
print('This may take a few minutes...\n')

# Use only datasets that loaded successfully
available_datasets = list(datasets.keys())[:3]  # Limit to first 3 for demo
limited_datasets = {k: datasets[k] for k in available_datasets}

evaluator_limited = CrossDatasetEvaluator(limited_datasets)
results = evaluator_limited.run_cross_evaluation(['logistic', 'random_forest'])

print('\n✅ Cross-dataset evaluation completed!')

## 4. Transfer Learning Analysis

Analyze the transfer learning performance across different domains.

In [ ]:
# Create transfer learning matrix
def create_transfer_matrix(results, model_type='logistic'):
    """Create a matrix showing transfer learning performance."""
    datasets_list = list(limited_datasets.keys())
    n_datasets = len(datasets_list)
    
    # Initialize matrix
    transfer_matrix = np.zeros((n_datasets, n_datasets))
    
    for i, source in enumerate(datasets_list):
        for j, target in enumerate(datasets_list):
            key = f'{source}_to_{target}_{model_type}'
            if key in results:
                transfer_matrix[i, j] = results[key]['accuracy']
    
    return transfer_matrix, datasets_list

# Create matrices for different models
lr_matrix, dataset_names = create_transfer_matrix(results, 'logistic')
rf_matrix, _ = create_transfer_matrix(results, 'random_forest')

# Visualize transfer learning matrices
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Logistic Regression
im1 = axes[0].imshow(lr_matrix, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[0].set_title('Transfer Learning: Logistic Regression')
axes[0].set_xlabel('Target Dataset')
axes[0].set_ylabel('Source Dataset')
axes[0].set_xticks(range(len(dataset_names)))
axes[0].set_yticks(range(len(dataset_names)))
axes[0].set_xticklabels(dataset_names, rotation=45)
axes[0].set_yticklabels(dataset_names)

# Add text annotations
for i in range(len(dataset_names)):
    for j in range(len(dataset_names)):
        text = axes[0].text(j, i, f'{lr_matrix[i, j]:.3f}',
                           ha='center', va='center', color='black')

# Random Forest
im2 = axes[1].imshow(rf_matrix, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[1].set_title('Transfer Learning: Random Forest')
axes[1].set_xlabel('Target Dataset')
axes[1].set_ylabel('Source Dataset')
axes[1].set_xticks(range(len(dataset_names)))
axes[1].set_yticks(range(len(dataset_names)))
axes[1].set_xticklabels(dataset_names, rotation=45)
axes[1].set_yticklabels(dataset_names)

# Add text annotations
for i in range(len(dataset_names)):
    for j in range(len(dataset_names)):
        text = axes[1].text(j, i, f'{rf_matrix[i, j]:.3f}',
                           ha='center', va='center', color='black')

# Add colorbar
fig.colorbar(im1, ax=axes, shrink=0.6, label='Accuracy')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze transfer learning patterns
def analyze_transfer_patterns(transfer_matrix, dataset_names):
    """Analyze patterns in transfer learning performance."""
    n_datasets = len(dataset_names)
    
    # Within-domain performance (diagonal)
    within_domain = np.diag(transfer_matrix)
    
    # Cross-domain performance (off-diagonal)
    cross_domain = []
    for i in range(n_datasets):
        for j in range(n_datasets):
            if i != j:
                cross_domain.append(transfer_matrix[i, j])
    
    cross_domain = np.array(cross_domain)
    
    # Calculate transfer learning metrics
    avg_within = within_domain.mean()
    avg_cross = cross_domain.mean()
    transfer_gap = avg_within - avg_cross
    
    print(f'Average within-domain accuracy: {avg_within:.4f}')
    print(f'Average cross-domain accuracy: {avg_cross:.4f}')
    print(f'Transfer gap: {transfer_gap:.4f}')
    
    # Best and worst transfers
    best_transfer_idx = np.unravel_index(np.argmax(transfer_matrix * (1 - np.eye(n_datasets))), 
                                        transfer_matrix.shape)
    worst_transfer_idx = np.unravel_index(np.argmin(transfer_matrix * (1 - np.eye(n_datasets)) + np.eye(n_datasets)), 
                                         transfer_matrix.shape)
    
    print(f'\nBest transfer: {dataset_names[best_transfer_idx[0]]} → {dataset_names[best_transfer_idx[1]]} ({transfer_matrix[best_transfer_idx]:.4f})')
    print(f'Worst transfer: {dataset_names[worst_transfer_idx[0]]} → {dataset_names[worst_transfer_idx[1]]} ({transfer_matrix[worst_transfer_idx]:.4f})')
    
    return {
        'avg_within': avg_within,
        'avg_cross': avg_cross,
        'transfer_gap': transfer_gap,
        'within_domain': within_domain,
        'cross_domain': cross_domain
    }

# Analyze both models
print('LOGISTIC REGRESSION TRANSFER ANALYSIS:')
lr_analysis = analyze_transfer_patterns(lr_matrix, dataset_names)

print('\nRANDOM FOREST TRANSFER ANALYSIS:')
rf_analysis = analyze_transfer_patterns(rf_matrix, dataset_names)

## 5. Feature Analysis Across Domains

Analyze how features transfer across different domains using XAI techniques.

In [ ]:
# Analyze feature importance across domains
def analyze_cross_domain_features(evaluator, dataset_names, model_type='logistic'):
    """Analyze feature importance across different domains."""
    feature_importance_data = {}
    
    for dataset in dataset_names:
        model_key = f'{dataset}_{model_type}'
        if model_key in evaluator.models:
            model = evaluator.models[model_key]
            vectorizer = evaluator.vectorizers[model_key]
            
            # Extract feature importance
            importance_df = extract_feature_importance(model, vectorizer)
            feature_importance_data[dataset] = importance_df
    
    return feature_importance_data

# Get feature importance for each domain
feature_data = analyze_cross_domain_features(evaluator_limited, dataset_names)

print('Feature importance analysis completed!')
for dataset, importance_df in feature_data.items():
    print(f'\n{dataset.upper()} - Top 5 features:')
    print(importance_df.head()[['Feature', 'Importance']].to_string(index=False))

In [ ]:
# Visualize feature overlap across domains
def visualize_feature_overlap(feature_data, top_n=20):
    """Visualize feature overlap across domains."""
    # Get top features for each dataset
    top_features = {}
    for dataset, importance_df in feature_data.items():
        top_features[dataset] = set(importance_df.head(top_n)['Feature'].tolist())
    
    # Calculate pairwise overlaps
    datasets_list = list(top_features.keys())
    n_datasets = len(datasets_list)
    overlap_matrix = np.zeros((n_datasets, n_datasets))
    
    for i, dataset1 in enumerate(datasets_list):
        for j, dataset2 in enumerate(datasets_list):
            if i == j:
                overlap_matrix[i, j] = len(top_features[dataset1])
            else:
                overlap = len(top_features[dataset1].intersection(top_features[dataset2]))
                overlap_matrix[i, j] = overlap
    
    # Visualize overlap matrix
    plt.figure(figsize=(10, 8))
    im = plt.imshow(overlap_matrix, cmap='Blues')
    plt.title(f'Feature Overlap Matrix (Top {top_n} features)')
    plt.xlabel('Dataset')
    plt.ylabel('Dataset')
    plt.xticks(range(n_datasets), datasets_list, rotation=45)
    plt.yticks(range(n_datasets), datasets_list)
    
    # Add text annotations
    for i in range(n_datasets):
        for j in range(n_datasets):
            text = plt.text(j, i, f'{int(overlap_matrix[i, j])}',
                           ha='center', va='center', color='white' if overlap_matrix[i, j] > overlap_matrix.max()/2 else 'black')
    
    plt.colorbar(im, label='Number of Overlapping Features')
    plt.tight_layout()
    plt.show()
    
    return overlap_matrix, datasets_list

# Visualize feature overlap
overlap_matrix, datasets_list = visualize_feature_overlap(feature_data)

## 6. Domain Adaptation Techniques

Implement simple domain adaptation techniques to improve transfer learning.

In [ ]:
class DomainAdaptationTransformer(BaseEstimator, TransformerMixin):
    """Simple domain adaptation using feature selection and weighting."""
    
    def __init__(self, adaptation_method='feature_selection', top_k=1000):
        self.adaptation_method = adaptation_method
        self.top_k = top_k
        self.selected_features = None
        self.feature_weights = None
    
    def fit(self, X_source, y_source, X_target=None):
        """Fit the domain adaptation transformer."""
        if self.adaptation_method == 'feature_selection':
            # Select features based on source domain importance
            from sklearn.feature_selection import SelectKBest, chi2
            selector = SelectKBest(chi2, k=self.top_k)
            selector.fit(X_source, y_source)
            self.selected_features = selector.get_support()
        
        elif self.adaptation_method == 'feature_weighting' and X_target is not None:
            # Weight features based on source-target similarity
            source_mean = np.array(X_source.mean(axis=0)).flatten()
            target_mean = np.array(X_target.mean(axis=0)).flatten()
            
            # Calculate feature similarity (inverse of difference)
            feature_diff = np.abs(source_mean - target_mean)
            self.feature_weights = 1 / (1 + feature_diff)
        
        return self
    
    def transform(self, X):
        """Transform the input data."""
        if self.adaptation_method == 'feature_selection' and self.selected_features is not None:
            return X[:, self.selected_features]
        
        elif self.adaptation_method == 'feature_weighting' and self.feature_weights is not None:
            return X.multiply(self.feature_weights)
        
        return X

# Test domain adaptation
def test_domain_adaptation(source_dataset, target_dataset, adaptation_method='feature_selection'):
    """Test domain adaptation between two datasets."""
    # Get data
    source_train = datasets[source_dataset]['train']
    target_test = datasets[target_dataset]['test']
    
    # Vectorize
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
    X_source = vectorizer.fit_transform(source_train['text'])
    y_source = source_train['label']
    X_target = vectorizer.transform(target_test['text'])
    y_target = target_test['label']
    
    # Baseline model (no adaptation)
    baseline_model = LogisticRegression(random_state=42, max_iter=1000)
    baseline_model.fit(X_source, y_source)
    baseline_acc = baseline_model.score(X_target, y_target)
    
    # Domain adaptation model
    adapter = DomainAdaptationTransformer(adaptation_method=adaptation_method)
    
    if adaptation_method == 'feature_weighting':
        adapter.fit(X_source, y_source, X_target)
    else:
        adapter.fit(X_source, y_source)
    
    X_source_adapted = adapter.transform(X_source)
    X_target_adapted = adapter.transform(X_target)
    
    adapted_model = LogisticRegression(random_state=42, max_iter=1000)
    adapted_model.fit(X_source_adapted, y_source)
    adapted_acc = adapted_model.score(X_target_adapted, y_target)
    
    print(f'{source_dataset} → {target_dataset}:')
    print(f'  Baseline accuracy: {baseline_acc:.4f}')
    print(f'  Adapted accuracy ({adaptation_method}): {adapted_acc:.4f}')
    print(f'  Improvement: {adapted_acc - baseline_acc:.4f}')
    
    return baseline_acc, adapted_acc

# Test domain adaptation on available datasets
print('Testing Domain Adaptation:')
print('=' * 40)

adaptation_results = {}
for source in dataset_names:
    for target in dataset_names:
        if source != target:
            baseline, adapted = test_domain_adaptation(source, target, 'feature_selection')
            adaptation_results[f'{source}_to_{target}'] = {
                'baseline': baseline,
                'adapted': adapted,
                'improvement': adapted - baseline
            }
            print()

## 7. Comprehensive Analysis and Insights

Summarize findings and provide insights for academic understanding.

In [ ]:
# Create comprehensive analysis summary
def create_analysis_summary(lr_analysis, rf_analysis, adaptation_results, overlap_matrix, dataset_names):
    """Create a comprehensive analysis summary."""
    
    print('=' * 60)
    print('COMPREHENSIVE CROSS-DATASET ANALYSIS SUMMARY')
    print('=' * 60)
    
    # 1. Transfer Learning Performance
    print('1. TRANSFER LEARNING PERFORMANCE:')
    print('-' * 40)
    print(f'Logistic Regression:')
    print(f'  Within-domain accuracy: {lr_analysis["avg_within"]:.4f}')
    print(f'  Cross-domain accuracy: {lr_analysis["avg_cross"]:.4f}')
    print(f'  Transfer gap: {lr_analysis["transfer_gap"]:.4f}')
    
    print(f'Random Forest:')
    print(f'  Within-domain accuracy: {rf_analysis["avg_within"]:.4f}')
    print(f'  Cross-domain accuracy: {rf_analysis["avg_cross"]:.4f}')
    print(f'  Transfer gap: {rf_analysis["transfer_gap"]:.4f}')
    
    # 2. Feature Analysis
    print('2. FEATURE OVERLAP ANALYSIS:')
    print('-' * 40)
    avg_overlap = np.mean(overlap_matrix[np.triu_indices_from(overlap_matrix, k=1)])
    max_overlap = np.max(overlap_matrix[np.triu_indices_from(overlap_matrix, k=1)])
    min_overlap = np.min(overlap_matrix[np.triu_indices_from(overlap_matrix, k=1)])
    
    print(f'Average feature overlap: {avg_overlap:.1f} features')
    print(f'Maximum feature overlap: {max_overlap:.0f} features')
    print(f'Minimum feature overlap: {min_overlap:.0f} features')
    
    # 3. Domain Adaptation Results
    print('3. DOMAIN ADAPTATION RESULTS:')
    print('-' * 40)
    improvements = [result['improvement'] for result in adaptation_results.values()]
    avg_improvement = np.mean(improvements)
    best_improvement = max(improvements)
    worst_improvement = min(improvements)
    
    print(f'Average improvement: {avg_improvement:.4f}')
    print(f'Best improvement: {best_improvement:.4f}')
    print(f'Worst improvement: {worst_improvement:.4f}')
    
    # 4. Academic Insights
    print('4. ACADEMIC INSIGHTS:')
    print('-' * 40)
    
    print('Key Findings:')
    if lr_analysis['transfer_gap'] > 0.1:
        print('• Significant transfer gap observed - models struggle with domain shift')
    else:
        print('• Good transfer performance - models generalize well across domains')
    
    if avg_overlap < 5:
        print('• Low feature overlap suggests distinct domain vocabularies')
    else:
        print('• Moderate feature overlap indicates shared domain characteristics')
    
    if avg_improvement > 0.01:
        print('• Domain adaptation techniques show promise for improvement')
    else:
        print('• Limited benefit from simple domain adaptation techniques')
    
    print('\nRecommendations for Future Work:')
    print('• Investigate advanced domain adaptation techniques (DANN, CORAL)')
    print('• Explore pre-trained language models for better transfer learning')
    print('• Analyze semantic similarity between domains')
    print('• Implement multi-source domain adaptation')
    
    return {
        'lr_analysis': lr_analysis,
        'rf_analysis': rf_analysis,
        'feature_overlap': {
            'avg': avg_overlap,
            'max': max_overlap,
            'min': min_overlap
        },
        'adaptation': {
            'avg_improvement': avg_improvement,
            'best_improvement': best_improvement,
            'worst_improvement': worst_improvement
        }
    }

# Generate comprehensive summary
summary = create_analysis_summary(lr_analysis, rf_analysis, adaptation_results, overlap_matrix, dataset_names)

## 8. Academic Exercises and Questions

Practice questions and exercises for deeper understanding.

In [ ]:
# Academic exercises
print('=' * 60)
print('ACADEMIC EXERCISES AND DISCUSSION QUESTIONS')
print('=' * 60)

exercises = [
    {
        'title': 'Exercise 1: Transfer Learning Analysis',
        'questions': [
            'Why do some domain pairs transfer better than others?',
            'What factors contribute to the transfer gap?',
            'How does model complexity affect transfer performance?'
        ]
    },
    {
        'title': 'Exercise 2: Feature Analysis',
        'questions': [
            'What does feature overlap tell us about domain similarity?',
            'How can we identify domain-specific vs. universal features?',
            'What role do stop words play in cross-domain transfer?'
        ]
    },
    {
        'title': 'Exercise 3: Domain Adaptation',
        'questions': [
            'Why might simple feature selection help with domain adaptation?',
            'What are the limitations of the implemented adaptation techniques?',
            'How could we improve domain adaptation performance?'
        ]
    },
    {
        'title': 'Exercise 4: Practical Applications',
        'questions': [
            'In what real-world scenarios would cross-dataset evaluation be important?',
            'How would you design a system that adapts to new domains?',
            'What metrics beyond accuracy should we consider for transfer learning?'
        ]
    }
]

for i, exercise in enumerate(exercises, 1):
    print(f'{exercise["title"]}:')
    print('-' * 40)
    for j, question in enumerate(exercise['questions'], 1):
        print(f'{j}. {question}')
    print()

print('IMPLEMENTATION CHALLENGES:')
print('-' * 40)
challenges = [
    'Implement adversarial domain adaptation',
    'Add semantic similarity metrics between domains',
    'Create ensemble methods for multi-source transfer',
    'Develop active learning for domain adaptation',
    'Implement gradual domain shift simulation'
]

for i, challenge in enumerate(challenges, 1):
    print(f'{i}. {challenge}')

print('\n✅ Cross-dataset evaluation and transfer learning analysis completed!')
print('📊 Review the generated visualizations and analysis results.')
print('🎓 Use the exercises above to deepen your understanding of transfer learning.')